### MLMMI Exercise 1: Lightweight Model Management System

Build a model registry for Titanic dataset with:
1. Dataset versioning (3 versions)
2. Model registration (10+ models)
3. Lineage tracking (3+ models with derivation)
4. Budget-constrained selection

---

In [1]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"Python path: {sys.path}")

Python executable: /usr/local/bin/python3
Python version: 3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
Python path: ['/Users/nirvanjhurree19/1_Model_management', '/Library/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload', '', '/Users/nirvanjhurree19/Library/Python/3.13/lib/python/site-packages', '/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages']


In [2]:
# Import dependencies and configure paths

# Development-only SSL bypass
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import pandas as pd
import numpy as np
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for imports
import sys
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import modules
from utils.data_loader import load_titanic_openml, save_raw_dataset
from utils.dataset_store import DatasetStore
from utils.model_registry import ModelRegistry
from utils.lineage import create_lineage_entry
from scripts.preprocess_v1 import create_ds_v1
from scripts.preprocess_v2 import create_ds_v2
from scripts.preprocess_v3 import create_ds_v3
from scripts.select_model import (
    select_best_under_constraint,
    training_time_constraint,
    accuracy_score_fn
)

# Configure paths
BASE_DIR = project_root
DATA_DIR = BASE_DIR / "data"
RAW_DATA_PATH = DATA_DIR / "raw" / "titanic.csv"
REGISTRY_DIR = BASE_DIR / "registry"
MODELS_DIR = BASE_DIR / "models"

print(f"Project root: {project_root}")
print(f"Base directory: {BASE_DIR}")

Project root: /Users/nirvanjhurree19/1_Model_management
Base directory: /Users/nirvanjhurree19/1_Model_management


### Task 1: Dataset Store & Versioning (3.0 pts)

Create 3 distinct dataset versions with different preprocessing strategies.

In [3]:
# Load raw dataset from OpenML
print("Loading Titanic dataset from OpenML (ID: 40945)...")
raw_df = load_titanic_openml(cache_dir=DATA_DIR / "raw")
print(f"Loaded dataset: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
print(f"Target distribution:\n{raw_df['survived'].value_counts().sort_index()}")

# Save raw data for reproducibility
save_raw_dataset(raw_df, RAW_DATA_PATH)

Loading Titanic dataset from OpenML (ID: 40945)...
Loaded dataset: 1309 rows, 14 columns
Target distribution:
survived
0    809
1    500
Name: count, dtype: int64
Saved raw dataset to: /Users/nirvanjhurree19/1_Model_management/data/raw/titanic.csv


PosixPath('/Users/nirvanjhurree19/1_Model_management/data/raw/titanic.csv')

In [4]:
# Create 3 dataset versions
print("\n Creating dataset versions...")

# Version 1: Basic preprocessing
print("\n Creating ds_v1 (basic)...")
v1_meta = create_ds_v1(raw_df, DATA_DIR / "ds_v1")

# Version 2: Advanced preprocessing  
print("Creating ds_v2 (advanced)...")
v2_meta = create_ds_v2(raw_df, DATA_DIR / "ds_v2")

# Version 3: Minimal preprocessing
print("Creating ds_v3 (minimal)...")
v3_meta = create_ds_v3(raw_df, DATA_DIR / "ds_v3")

print(f"\n Created 3 dataset versions:")
for meta in [v1_meta, v2_meta, v3_meta]:
    print(f" {meta['dataset_version_id']}: {meta['schema']['shape'][1]} features, "
          f"{len(meta['preprocessing_steps'])} preprocessing steps")


 Creating dataset versions...

 Creating ds_v1 (basic)...
Created dataset version 'ds_v1' at /Users/nirvanjhurree19/1_Model_management/data/ds_v1
Creating ds_v2 (advanced)...
Created dataset version 'ds_v2' at /Users/nirvanjhurree19/1_Model_management/data/ds_v2
Creating ds_v3 (minimal)...
Created dataset version 'ds_v3' at /Users/nirvanjhurree19/1_Model_management/data/ds_v3

 Created 3 dataset versions:
 ds_v1: 8 features, 5 preprocessing steps
 ds_v2: 11 features, 6 preprocessing steps
 ds_v3: 8 features, 6 preprocessing steps


### Task 2: Model Registry (3.0 pts)

Train and register 10+ models with comprehensive metadata tracking.

In [5]:
# Initialise model registry
registry = ModelRegistry(REGISTRY_DIR, MODELS_DIR)
print(f"Initialised model registry at {REGISTRY_DIR}")

Initialised model registry at /Users/nirvanjhurree19/1_Model_management/registry


In [6]:
# Helper function to train and register a model
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

def train_and_register_model(
    model_id: str,
    model_instance,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    dataset_version: str,
    lineage: dict = None,
    notes: str = ""
) -> dict:
    """Train a model and register it with full metadata."""
    
    # Measure training time
    start_train = time.perf_counter()
    model_instance.fit(X_train, y_train)
    training_time = time.perf_counter() - start_train
    
    # Evaluate on test set
    y_pred = model_instance.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Measure inference time (using registry's method)
    inference_time = registry._measure_inference_time(model_instance, X_test)
    
    # Prepare metadata
    metrics = {"accuracy": accuracy}
    hyperparams = model_instance.get_params()
    
    # Register in model registry
    entry = registry.register_model(
        model=model_instance,
        model_id=model_id,
        algorithm=model_instance.__class__.__name__,
        hyperparameters=hyperparams,
        metrics=metrics,
        training_time_sec=training_time,
        inference_time_sec=inference_time,
        dataset_version=dataset_version,
        lineage=lineage,
        notes=notes
    )
    
    return entry

In [7]:
# Load dataset version 1 for training (we'll use v1 for most models)
print("Loading dataset version 1 for model training...")
X_train = pd.read_csv(DATA_DIR / "ds_v1" / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "ds_v1" / "y_train.csv").squeeze()
X_test = pd.read_csv(DATA_DIR / "ds_v1" / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "ds_v1" / "y_test.csv").squeeze()

print(f"Training  {X_train.shape}, Test  {X_test.shape}")

Loading dataset version 1 for model training...
Training  (730, 7), Test  (313, 7)


In [8]:
# Train and register 10+ models
print("\n Training and registering 10+ models...\n")

models_config = [
    # === Baseline models (no lineage) ===
    {
        "model_id": "model_001",
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "dataset": "ds_v1",
        "notes": "Baseline logistic regression"
    },
    {
        "model_id": "model_002",
        "model": RandomForestClassifier(n_estimators=10, random_state=42),
        "dataset": "ds_v1",
        "notes": "Baseline random forest (10 trees)"
    },
    {
        "model_id": "model_003",
        "model": SVC(kernel="linear", random_state=42),
        "dataset": "ds_v1",
        "notes": "Linear SVM baseline"
    },
    {
        "model_id": "model_004",
        "model": GaussianNB(),
        "dataset": "ds_v1",
        "notes": "Naive Bayes baseline"
    },
    {
        "model_id": "model_005",
        "model": DecisionTreeClassifier(max_depth=5, random_state=42),
        "dataset": "ds_v1",
        "notes": "Decision tree with depth limit"
    },
    
    # === Lineage examples (derived from baselines) ===
    {
        "model_id": "model_006",
        "model": LogisticRegression(C=0.1, max_iter=1000, random_state=42),
        "dataset": "ds_v1",
        "lineage": create_lineage_entry(
            parent_model_id="model_001",
            relationship_type="hyperparameter_tuned",
            description="Tuned regularization C=0.1 from model_001 baseline",
            hyperparameter_changes={"C": {"old": 1.0, "new": 0.1}}
        ),
        "notes": "Hyperparameter-tuned logistic regression"
    },
    {
        "model_id": "model_007",
        "model": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
        "dataset": "ds_v1",
        "lineage": create_lineage_entry(
            parent_model_id="model_002",
            relationship_type="hyperparameter_tuned",
            description="Increased trees to 100 and added depth limit from model_002",
            hyperparameter_changes={
                "n_estimators": {"old": 10, "new": 100},
                "max_depth": {"old": None, "new": 10}
            }
        ),
        "notes": "Enhanced random forest with more trees"
    },
    {
        "model_id": "model_008",
        "model": SVC(kernel="rbf", gamma="scale", random_state=42),
        "dataset": "ds_v1",
        "lineage": create_lineage_entry(
            parent_model_id="model_003",
            relationship_type="algorithm_variant",
            description="Changed kernel from linear to RBF from model_003",
            algorithm_change={"kernel": {"old": "linear", "new": "rbf"}}
        ),
        "notes": "RBF kernel SVM variant"
    },
    
    # === Models on different dataset versions (lineage via data) ===
    {
        "model_id": "model_009",
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "dataset": "ds_v2",  # Different dataset version
        "lineage": create_lineage_entry(
            parent_model_id="model_001",
            relationship_type="retrained_on_new_data",
            description="Retrained model_001 architecture on ds_v2 (advanced preprocessing)",
            dataset_change="ds_v1 -> ds_v2: added feature engineering, one-hot encoding"
        ),
        "notes": "Same algorithm, different dataset version"
    },
    {
        "model_id": "model_010",
        "model": GradientBoostingClassifier(n_estimators=50, random_state=42),
        "dataset": "ds_v1",
        "notes": "Gradient boosting baseline"
    },
    {
        "model_id": "model_011",
        "model": KNeighborsClassifier(n_neighbors=5),
        "dataset": "ds_v1",
        "notes": "KNN baseline"
    }
]

# Train and register each model
registered_models = []
for config in models_config:
    # Load appropriate dataset version
    ds_version = config["dataset"]
    if ds_version == "ds_v1":
        X_tr, y_tr = X_train, y_train
        X_te, y_te = X_test, y_test
    else:
        # Load other dataset versions
        X_tr = pd.read_csv(DATA_DIR / ds_version / "X_train.csv")
        y_tr = pd.read_csv(DATA_DIR / ds_version / "y_train.csv").squeeze()
        X_te = pd.read_csv(DATA_DIR / ds_version / "X_test.csv")
        y_te = pd.read_csv(DATA_DIR / ds_version / "y_test.csv").squeeze()
    
    # Train and register
    entry = train_and_register_model(
        model_id=config["model_id"],
        model_instance=config["model"],
        X_train=X_tr, y_train=y_tr,
        X_test=X_te, y_test=y_te,
        dataset_version=ds_version,
        lineage=config.get("lineage"),
        notes=config.get("notes", "")
    )
    registered_models.append(entry)

print(f"\n Registered {len(registered_models)} models in total")


 Training and registering 10+ models...

Registered model 'model_001' (accuracy: 0.7796)
Registered model 'model_002' (accuracy: 0.7540)
Registered model 'model_003' (accuracy: 0.7668)
Registered model 'model_004' (accuracy: 0.7636)
Registered model 'model_005' (accuracy: 0.7796)
Registered model 'model_006' (accuracy: 0.7859)
Registered model 'model_007' (accuracy: 0.7827)
Registered model 'model_008' (accuracy: 0.6677)
Registered model 'model_009' (accuracy: 0.8092)
Registered model 'model_010' (accuracy: 0.8115)
Registered model 'model_011' (accuracy: 0.6741)

 Registered 11 models in total


### Task 3: Lineage Tracking (2.0 pts)

Verify that ≥3 models have non-trivial lineage relationships.

In [9]:
# List models with lineage
print("Models with lineage tracking:\n")

models_with_lineage = [
    m for m in registry.list_models() 
    if m.get("lineage") and m["lineage"].get("parent_model_id")
]

for model in models_with_lineage:
    lineage = model["lineage"]
    print(f"{model['model_id']} ({model['algorithm']})")
    print(f"Derived from: {lineage['parent_model_id']}")
    print(f"Relationship: {lineage['relationship_type']}")
    print(f"Description: {lineage['description']}")
    if lineage.get('hyperparameter_changes'):
        print(f"HP changes: {lineage['hyperparameter_changes']}")
    print()

print(f"Found {len(models_with_lineage)} models with lineage (requirement: ≥3)")

Models with lineage tracking:

model_006 (LogisticRegression)
Derived from: model_001
Relationship: hyperparameter_tuned
Description: Tuned regularization C=0.1 from model_001 baseline
HP changes: {'C': {'old': 1.0, 'new': 0.1}}

model_007 (RandomForestClassifier)
Derived from: model_002
Relationship: hyperparameter_tuned
Description: Increased trees to 100 and added depth limit from model_002
HP changes: {'n_estimators': {'old': 10, 'new': 100}, 'max_depth': {'old': None, 'new': 10}}

model_008 (SVC)
Derived from: model_003
Relationship: algorithm_variant
Description: Changed kernel from linear to RBF from model_003

model_009 (LogisticRegression)
Derived from: model_001
Relationship: retrained_on_new_data
Description: Retrained model_001 architecture on ds_v2 (advanced preprocessing)

Found 4 models with lineage (requirement: ≥3)


### Task 4: Budget-Constrained Selection (2.0 pts)

Select best model under a fixed constraint with clear documentation.

In [10]:
# Define the constraint and selection logic
print("Model Selection Under Budget Constraint\n")
print("Constraint: Maximize accuracy subject to training_time < 1.5 seconds")
print("Rationale: Balance model performance with training efficiency for iterative development\n")

# Execute selection
best_model = select_best_under_constraint(
    registry_dir=REGISTRY_DIR,
    constraint_fn=training_time_constraint(max_seconds=1.5),
    score_fn=accuracy_score_fn,
    constraint_description="maximize accuracy with training_time < 1.5 seconds"
)

# Document the selection
print(f"\n Selection Documentation:")
print(f"   - Constraint applied: training_time_sec < 1.5s")
print(f"   - Selection rule: argmax(accuracy) among feasible models")
print(f"   - Selected model: {best_model['model_id']}")
print(f"   - Justification: Highest accuracy ({best_model['metrics']['accuracy']:.4f}) "
      f"while meeting training time budget ({best_model['training_time_sec']:.2f}s < 1.5s)")
if best_model.get('lineage', {}).get('description'):
    print(f"   - Lineage context: {best_model['lineage']['description']}")

Model Selection Under Budget Constraint

Constraint: Maximize accuracy subject to training_time < 1.5 seconds
Rationale: Balance model performance with training efficiency for iterative development

   Constraint: maximize accuracy with training_time < 1.5 seconds
   Candidates: 10 models satisfy constraint
   Selected: model_010
   Score: 0.8115
   Accuracy: 0.8115
   Training time: 0.08s
   Inference time: 0.0000s/sample

 Selection Documentation:
   - Constraint applied: training_time_sec < 1.5s
   - Selection rule: argmax(accuracy) among feasible models
   - Selected model: model_010
   - Justification: Highest accuracy (0.8115) while meeting training time budget (0.08s < 1.5s)
